# 8.12 裁判偏好的是新資訊，還是更多字？


短版「答案是4。」與把它重複十次的長版，有相同事實。若裁判總選後者，可能只是喜歡篇幅。這裡的**長度偏差**指沒有相應內容收益，評分卻不合理地受字數影響。

先控制內容，再改長度。以下故意用字元數當分數，模擬這個漏洞。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
short = "答案是4。"
long = short * 10
candidates = [short, long]
lengths = [len(text) for text in candidates]
chosen = max(candidates, key=len)
print("字元數", lengths)
print("長度裁判選長版", chosen == long)
print("不同句子數", len(set(long.split("。")[:-1])))

字元數是`[5,50]`，`max(...,key=len)`必定選長版。按句號切開、去掉句末空字串、用`set`去重後，不同句子數仍是1。這些都是字串操作，沒有理解語義；字元數也不同於模型切成的token數。

再對照真正有用的長版：「答案是4；兩組各兩個物件合起來共有四個。」它增加了可核對的分組關係。裁判選這版時，理由可以是新解釋，而非字數。不能為排除長度偏差，把任務所需的說明一律刪掉。

練習先把重複次數改2，預測字元數變`[5,10]`而不同句子仍1；再把長版換成上面的有用解釋。它也只有一個句號結尾，「不同句子數」仍1，所以這個粗計數只能幫本例看重複，不能充當通用資訊分數。保留句末句號，因為程式的`[:-1]`假定最後一項是空字串。
